# YOLO Model Training for Manga Text Detection

This notebook walks through the process of training a custom YOLO model for detecting text bubbles in manga.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from ultralytics import YOLO
import torch
import yaml
from pathlib import Path

## 2. Check GPU Availability

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 3. Prepare Dataset

Make sure your dataset is organized in YOLO format:
```
data/
  processed/
    train/
      images/
      labels/
    val/
      images/
      labels/
```

In [ ]:
# Load dataset configuration
data_yaml = '../training/datasets/manga_text_detection.yaml'

with open(data_yaml, 'r') as f:
    config = yaml.safe_load(f)
    print("Dataset configuration:")
    print(f"  Classes: {config['nc']}")
    print(f"  Names: {config['names']}")

## 4. Initialize Model

In [ ]:
# Choose model size
model_size = 'yolo11m'  # Options: yolo11n, yolo11s, yolo11m, yolo11l, yolo11x

# Load pretrained model
model = YOLO(f'{model_size}.pt')
print(f"Loaded {model_size} model")

## 5. Train Model

In [ ]:
# Training configuration
results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=50,
    save=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    workers=8,
    project='../models/checkpoints',
    name='manga_text_detection',
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    verbose=True,
    seed=42,
    deterministic=True,
    single_cls=False,
    rect=False,
    cos_lr=False,
    close_mosaic=10,
    resume=False,
    amp=True,
    fraction=1.0,
    profile=False,
    freeze=None,
)

## 6. Evaluate Model

In [ ]:
# Validate on validation set
metrics = model.val()

print("\nEvaluation Results:")
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

## 7. Test on Sample Images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Run inference on a test image
test_image = '../data/raw/test_page.jpg'  # Update with your image

results = model.predict(test_image, save=True, conf=0.25)

# Display results
for result in results:
    img = Image.open(test_image)
    plt.figure(figsize=(12, 8))
    plt.imshow(result.plot())
    plt.axis('off')
    plt.title('Detection Results')
    plt.show()
    
    # Print detections
    print(f"\nDetected {len(result.boxes)} text regions")
    for box in result.boxes:
        class_id = int(box.cls)
        confidence = float(box.conf)
        print(f"  Class: {config['names'][class_id]}, Confidence: {confidence:.2f}")

## 8. Export Model

In [ ]:
# Export to different formats
export_dir = Path('../models/exported')
export_dir.mkdir(parents=True, exist_ok=True)

# Export to ONNX (for cross-platform deployment)
model.export(format='onnx', dynamic=True)
print("Model exported to ONNX format")

# Export to TorchScript (for production)
model.export(format='torchscript')
print("Model exported to TorchScript format")

## 9. Save Training Summary

In [ ]:
import json
from datetime import datetime

summary = {
    'model': model_size,
    'dataset': data_yaml,
    'timestamp': datetime.now().isoformat(),
    'metrics': {
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr)
    }
}

with open('../models/checkpoints/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Training summary saved!")